# 02 · Silver layer: cleaned and standardized drivers

**Purpose.** Turn the raw Bronze strings into a typed, validated table with one row per driver. Every cleaning rule is checked against the data first, and its effect on the row count is reported.

**Inputs.** `workspace.car_insurance.claims_bronze` (from `01_bronze_layer`).

**Outputs.** `workspace.car_insurance.claims_silver`: 19 typed columns, one row per driver.

**Assumptions**

- `01_bronze_layer` has been run.
- Runs on serverless compute, where invalid casts raise errors. All conversions therefore use `try_cast`, which returns `NULL` for a value that cannot be converted, and those cases are counted.
- All work uses Spark DataFrames. Only small summary results are brought back for display.

**What Silver deliberately does not do.** It does not impute, scale or encode. In Phase 2 those steps were fitted on the training drivers only, and repeating them here on all 10,000 rows would leak test-set information into the models. They are fitted inside the Spark ML pipeline in `04_mllib_models`.

In [ ]:
# --- Configuration (identical in every Phase 3 notebook) ----------------------
# Free Edition workspaces come with a Unity Catalog catalog named `workspace`.
# If yours is named differently, change CATALOG here and in the other notebooks.
CATALOG = "workspace"
SCHEMA = "car_insurance"
VOLUME = "raw"

RAW_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CLAIMS_CSV = f"{RAW_DIR}/Car_Insurance_Claim.csv"
SPLIT_CSV = f"{RAW_DIR}/split_assignment.csv"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.claims_bronze"
SPLIT_BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.claims_split_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.claims_silver"
GOLD_FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.claims_gold_features"
GOLD_SUMMARY_TABLE = f"{CATALOG}.{SCHEMA}.claims_gold_summary"

# The 19 columns of the source file, in file order.
SOURCE_COLUMNS = [
    "ID", "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME",
    "CREDIT_SCORE", "VEHICLE_OWNERSHIP", "VEHICLE_YEAR", "MARRIED", "CHILDREN",
    "POSTAL_CODE", "ANNUAL_MILEAGE", "VEHICLE_TYPE", "SPEEDING_VIOLATIONS",
    "DUIS", "PAST_ACCIDENTS", "OUTCOME",
]

In [ ]:
from pyspark.sql import functions as F


def write_delta(df, table_name, comment):
    """Overwrite a managed Delta table so the notebook can be rerun top to bottom."""
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))
    spark.sql(f"COMMENT ON TABLE {table_name} IS '{comment}'")


def validate_table(table_name, expected_rows, expected_columns):
    """Read a table back from the catalog and compare it with what was written."""
    saved = spark.table(table_name)
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").first()
    rows, columns = saved.count(), len(saved.columns)
    passed = (detail["format"] == "delta" and rows == expected_rows
              and columns == expected_columns)
    return (table_name, detail["format"], expected_rows, rows,
            expected_columns, columns, "PASS" if passed else "FAIL")


VALIDATION_SCHEMA = ("table string, format string, expected_rows long, actual_rows long, "
                     "expected_columns int, actual_columns int, status string")


def show_validation(results):
    display(spark.createDataFrame(results, VALIDATION_SCHEMA))
    failed = [r[0] for r in results if r[-1] != "PASS"]
    assert not failed, f"Validation failed for: {failed}"

In [ ]:
# How each source column is treated.
CATEGORICAL_COLUMNS = ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION",
                       "INCOME", "VEHICLE_YEAR", "POSTAL_CODE", "VEHICLE_TYPE"]
CONTINUOUS_COLUMNS = ["CREDIT_SCORE", "ANNUAL_MILEAGE"]           # -> double
FLAG_COLUMNS = ["VEHICLE_OWNERSHIP", "MARRIED", "CHILDREN", "OUTCOME"]  # 0/1 -> int
COUNT_COLUMNS = ["SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]  # -> int
INTEGER_COLUMNS = FLAG_COLUMNS + COUNT_COLUMNS
NUMERIC_COLUMNS = ["ID"] + CONTINUOUS_COLUMNS + INTEGER_COLUMNS

# Category values seen in Phases 1 and 2. Anything else is reported as invalid.
EXPECTED_CATEGORIES = {
    "AGE": ["16-25", "26-39", "40-64", "65+"],
    "GENDER": ["female", "male"],
    "RACE": ["majority", "minority"],
    "DRIVING_EXPERIENCE": ["0-9y", "10-19y", "20-29y", "30y+"],
    "EDUCATION": ["high school", "none", "university"],
    "INCOME": ["middle class", "poverty", "upper class", "working class"],
    "VEHICLE_YEAR": ["after 2015", "before 2015"],
    "POSTAL_CODE": ["10238", "21217", "32765", "92101"],
    "VEHICLE_TYPE": ["sedan", "sports car"],
}

bronze = spark.table(BRONZE_TABLE)
bronze_rows = bronze.count()
print(f"Bronze rows: {bronze_rows:,}")

## 1. Inspect the Bronze data

The profile below covers the five things the cleaning rules depend on: malformed records, duplicates, missing values, values that cannot be converted to their intended type, and inconsistent category labels.

In [ ]:
malformed_rows = bronze.filter(F.col("_corrupt_record").isNotNull()).count()
duplicate_rows = bronze_rows - bronze.select(SOURCE_COLUMNS).distinct().count()
duplicate_ids = bronze_rows - bronze.select("ID").distinct().count()

print(f"Malformed records      : {malformed_rows:,}")
print(f"Exact duplicate rows   : {duplicate_rows:,}")
print(f"Repeated ID values     : {duplicate_ids:,}")

In [ ]:
# Missing values, and values that are present but not convertible to a number.
profile_row = bronze.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__missing") for c in SOURCE_COLUMNS]
    + [F.sum((F.col(c).isNotNull() & F.expr(f"try_cast({c} AS double)").isNull()).cast("int"))
        .alias(f"{c}__not_numeric") for c in NUMERIC_COLUMNS]
    + [F.sum((F.expr(f"try_cast({c} AS double)") % 1 != 0).cast("int"))
        .alias(f"{c}__not_whole") for c in ["ID"] + INTEGER_COLUMNS]
).first().asDict()

bronze_missing = {c: int(profile_row[f"{c}__missing"]) for c in SOURCE_COLUMNS}

display(
    spark.createDataFrame(
        [(c,
          "categorical" if c in CATEGORICAL_COLUMNS else "numeric",
          bronze_missing[c],
          int(profile_row.get(f"{c}__not_numeric") or 0),
          int(profile_row.get(f"{c}__not_whole") or 0))
         for c in SOURCE_COLUMNS],
        "column string, intended_type string, missing long, not_numeric long, not_whole_number long",
    )
)

In [ ]:
# Distinct labels per categorical column, exactly as stored (shows casing or spacing variants).
category_profile = bronze.select(
    [F.sort_array(F.collect_set(c)).alias(c) for c in CATEGORICAL_COLUMNS]
).first().asDict()

display(
    spark.createDataFrame(
        [(c, len(values), ", ".join(f"'{v}'" for v in values))
         for c, values in category_profile.items()],
        "column string, distinct_values int, values string",
    )
)

**What the profile shows**

- No malformed records, no duplicate rows and no repeated `ID`.
- Missing values occur only in `CREDIT_SCORE` (982) and `ANNUAL_MILEAGE` (957), about 10% each.
- Every numeric column converts cleanly, and the 0/1 and count columns hold whole numbers (stored as text such as `1.0`).
- Category labels are already consistent: no casing or spacing variants.

## 2. Cleaning rules

| # | Rule | Why | Effect on this data |
|---|---|---|---|
| 1 | Exclude records flagged in `_corrupt_record` | A row that did not parse cannot be trusted column by column; it stays in Bronze for inspection | 0 rows removed |
| 2 | Trim and lower-case category labels; empty text becomes `NULL` | Guarantees one spelling per category for encoding | No value changed |
| 3 | Cast `CREDIT_SCORE`, `ANNUAL_MILEAGE` to `double`; 0/1 flags and counts to `int`; `ID` to `bigint` | Bronze stores text; models and aggregations need numbers | 10 columns typed, 0 conversion failures |
| 4 | Keep `POSTAL_CODE` as a string | It is a label for one of four areas, not a quantity (same decision as Phase 1) | Stays categorical |
| 5 | Remove exact duplicate rows | A repeated driver would be counted twice in training and evaluation | 0 rows removed |
| 6 | Keep missing `CREDIT_SCORE` / `ANNUAL_MILEAGE` as `NULL` | Dropping them would lose about 18% of drivers, and imputing here would use test rows (see the note at the top) | Nulls preserved |

The Phase 1 relabelling of `AGE` and `DRIVING_EXPERIENCE` merges categories, which is a modelling choice, so it is applied in the Gold layer rather than here.

In [ ]:
# Rule 1: keep only records that parsed.
parsed = bronze.filter(F.col("_corrupt_record").isNull())
rows_after_rule_1 = parsed.count()


def clean_column(name):
    if name in CATEGORICAL_COLUMNS:            # rules 2 and 4
        return F.expr(f"nullif(lower(trim({name})), '')").alias(name)
    if name == "ID":                           # rule 3
        return F.expr("try_cast(ID AS bigint)").alias(name)
    if name in CONTINUOUS_COLUMNS:
        return F.expr(f"try_cast({name} AS double)").alias(name)
    # Flags and counts are written like "1.0": go through double, then to int.
    return F.expr(f"CAST(try_cast({name} AS double) AS int)").alias(name)


typed = parsed.select([clean_column(name) for name in SOURCE_COLUMNS])

# Rule 5: remove exact duplicates.
silver_df = typed.dropDuplicates()

# Missing values among the parsed Bronze rows, to confirm later that no cast created a NULL.
parsed_missing = {
    name: int(count) for name, count in parsed.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in SOURCE_COLUMNS]
    ).first().asDict().items()
}

silver_df.printSchema()

## 3. Write the Silver Delta table

In [ ]:
expected_silver_rows = silver_df.count()

write_delta(
    silver_df,
    SILVER_TABLE,
    "Typed and validated car insurance drivers, one row per ID. Missing values are kept as NULL.",
)
print("Wrote", SILVER_TABLE)

## 4. Quality checks on the saved table

All checks below run against the table read back from the catalog. Each one counts the rows that break a rule; the notebook stops if any check fails.

In [ ]:
silver = spark.table(SILVER_TABLE)
silver_rows = silver.count()


def invalid_category(name):
    return F.col(name).isNull() | ~F.col(name).isin(EXPECTED_CATEGORIES[name])


checks = {
    "ID is missing": F.col("ID").isNull(),
    "OUTCOME is not 0 or 1": F.col("OUTCOME").isNull() | ~F.col("OUTCOME").isin(0, 1),
    "VEHICLE_OWNERSHIP / MARRIED / CHILDREN not 0 or 1":
        F.col("VEHICLE_OWNERSHIP").isNull() | ~F.col("VEHICLE_OWNERSHIP").isin(0, 1)
        | F.col("MARRIED").isNull() | ~F.col("MARRIED").isin(0, 1)
        | F.col("CHILDREN").isNull() | ~F.col("CHILDREN").isin(0, 1),
    "Count column missing or negative":
        F.col("SPEEDING_VIOLATIONS").isNull() | (F.col("SPEEDING_VIOLATIONS") < 0)
        | F.col("DUIS").isNull() | (F.col("DUIS") < 0)
        | F.col("PAST_ACCIDENTS").isNull() | (F.col("PAST_ACCIDENTS") < 0),
    "CREDIT_SCORE outside 0-1": (F.col("CREDIT_SCORE") < 0) | (F.col("CREDIT_SCORE") > 1),
    "ANNUAL_MILEAGE not positive": F.col("ANNUAL_MILEAGE") <= 0,
}
for name in CATEGORICAL_COLUMNS:
    checks[f"{name} has an unexpected or missing label"] = invalid_category(name)

check_names = list(checks)
failing = silver.select(
    [F.coalesce(F.sum(checks[n].cast("int")), F.lit(0)).alias(f"check_{i}")
     for i, n in enumerate(check_names)]
).first()

results = [(n, int(failing[f"check_{i}"])) for i, n in enumerate(check_names)]
results.append(("ID is repeated", silver_rows - silver.select("ID").distinct().count()))

display(
    spark.createDataFrame(
        [(n, count, "PASS" if count == 0 else "FAIL") for n, count in results],
        "check string, failing_rows long, status string",
    )
)
assert all(count == 0 for _, count in results), "A Silver quality check failed"

In [ ]:
# Before-and-after comparison of Bronze and Silver.
silver_missing = silver.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in SOURCE_COLUMNS]
).first().asDict()

string_columns = lambda df: sum(1 for _, dtype in df.select(SOURCE_COLUMNS).dtypes if dtype == "string")

comparison = [
    ("Rows", bronze_rows, silver_rows),
    ("Rows removed: malformed (rule 1)", 0, bronze_rows - rows_after_rule_1),
    ("Rows removed: exact duplicates (rule 5)", 0, rows_after_rule_1 - silver_rows),
    ("Source columns stored as text", string_columns(bronze), string_columns(silver)),
    ("Missing CREDIT_SCORE", bronze_missing["CREDIT_SCORE"], int(silver_missing["CREDIT_SCORE"])),
    ("Missing ANNUAL_MILEAGE", bronze_missing["ANNUAL_MILEAGE"], int(silver_missing["ANNUAL_MILEAGE"])),
    ("Missing values in all other columns",
     sum(v for c, v in bronze_missing.items() if c not in CONTINUOUS_COLUMNS),
     sum(int(v) for c, v in silver_missing.items() if c not in CONTINUOUS_COLUMNS)),
]

display(spark.createDataFrame(comparison, "metric string, bronze long, silver long"))

# A cast must never turn a present value into NULL (duplicates removed: counts can only drop).
introduced = {c: int(v) - parsed_missing[c] for c, v in silver_missing.items() if int(v) > parsed_missing[c]}
assert not introduced, f"A cast introduced NULLs: {introduced}"

In [ ]:
display(silver.limit(10))

## 5. Delta validation: read back and verify

In [ ]:
show_validation([
    validate_table(SILVER_TABLE, expected_silver_rows, len(SOURCE_COLUMNS)),
])
silver.printSchema()

**Result.** Silver keeps all 10,000 drivers. No row was removed and no value was imputed; the changes are type conversions and label standardization. The missing `CREDIT_SCORE` and `ANNUAL_MILEAGE` values are carried forward unchanged for the modelling pipeline to impute from training data.

Next: `03_gold_layer`.